# Baseline using matrix factorization

We use an implicit ALS model as our collaborative filtering baseline due to its efficiency on CPU and strong performance in implicit-feedback settings. It is used as the industry standard in many collabrative filtering settings Pairwise ranking approaches such as BPR were considered but found computationally expensive.

Note: implicit needs extra tools before it can be installed on PC, which is why I moved to colab.

In [1]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"

In [2]:
!pip install implicit

In [3]:
from google.colab import drive
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from implicit.als import AlternatingLeastSquares
import pickle
from collections import Counter
from torch.utils.data import Dataset,DataLoader
from implicit.evaluation import mean_average_precision_at_k
import random
import scipy.sparse as sp
np.random.seed(44)


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Import datasets

In [24]:

import os
import pickle
from pathlib import Path

def load_pkl(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

# 1. Define your base path relative to your Google Drive structure
# Replace 'Your_Project_Folder' with the actual folder name in your Drive
base_path = Path('/content/drive/MyDrive/project dl/datasets/processed')

# 2. Define paths using Path objects (handles / vs \ automatically)
train_sessions_path = base_path / 'train' / 'session_train.pkl'
val_sessions_path = base_path / 'val' / 'session_val.pkl'
test_sessions_path = base_path / 'val' / 'session_val.pkl'
# 3. Load the files
# Added a quick check to see if the files exist to avoid "File Not Found" errors
if train_sessions_path.exists():
    train_sessions = load_pkl(train_sessions_path)
    val_sessions = load_pkl(val_sessions_path)
    test_sessions= load_pkl(test_sessions_path)
    #train_360k = load_pkl(train_360k_path)
    #val_360k = load_pkl(val_360k_path)
    print("All datasets loaded successfully from Google Drive.")
else:
    print(f"Error: Could not find files at {base_path}. Please check your Drive folder name.")


All datasets loaded successfully from Google Drive.


In [42]:
with open(base_path /"track_vocab.pkl", "rb") as f:
    track_vocab = pickle.load(f)
with open(base_path /"artist_vocab.pkl", "rb") as f:
    artist_vocab = pickle.load(f)
with open(base_path /"user_vocab.pkl", "rb") as f:
    user_vocab = pickle.load(f)

In [ ]:
list(artist_vocab['artist2idx'])[-100]

# 1k dataset

In [6]:
train_sessions

,user_key,artist_key,track_key,session_start
global_session_id,,,,
user_000001_1,user_000001,"[10216, 10216, 10216, 109823, 30662, 113575, 1...","[13069633, 12523960, 11373798, 14448993, 11701...",2006-08-13 13:59:20+00:00
user_000001_10,user_000001,"[208731, 117439, 203929, 259085, 229495]","[7828701, 8173444, 3541483, 7730974, 4275928]",2006-08-21 17:55:52+00:00
user_000001_100,user_000001,"[255140, 255140, 30023, 30023, 30023, 30023, 3...","[2755454, 13184824, 4024722, 9342000, 9500276,...",2006-11-16 01:49:47+00:00
user_000001_1000,user_000001,"[25599, 25599, 25599, 25599, 25599, 25599, 255...","[1573967, 3824057, 3003535, 5015731, 14859334,...",2009-04-05 14:52:39+00:00
user_000001_1001,user_000001,"[25599, 25599, 25599, 25599, 25599, 25599, 255...","[11594693, 14837824, 1888883, 1573967, 3824057...",2009-04-06 17:38:45+00:00
...,...,...,...,...
user_001000_986,user_001000,"[176444, 176444, 176444, 176444, 176444, 17644...","[15231752, 10881675, 9552615, 16392812, 764557...",2009-04-30 05:34:00+00:00
user_001000_988,user_001000,"[176444, 176444, 176444, 176444, 176444, 17644...","[13614390, 3347068, 5141975, 2675771, 1847754,...",2009-05-01 05:24:10+00:00
user_001000_989,user_001000,"[21964, 21964, 21964, 21964, 21964, 21964]","[5886516, 11285291, 8991275, 9778064, 14251671...",2009-05-01 21:26:52+00:00


## Baseline 1: Random

In [7]:
def random_suggestions(train_df=train_sessions,k=10):
  all_tracks = df["track_key"].explode().unique()
  track_counts = all_tracks.value_counts()
  probs = track_counts / track_counts.sum()

  random_tracks = np.random.choice(
    probs.index,
    size=10,
    replace=False,
    p=probs.values
)
  return(random_tracks)


## Baseline 2: popularity suggestion

In [8]:
def popular_suggestions(train_df=train_sessions,k=10):
  popular_tracks = (
    train_df["track_key"]
    .explode()
    .value_counts()
  )
  top_k = popular_tracks.head(10)
  return(top_k)

## Baseline 3: Matrix factorization

In [29]:
df = train_sessions.explode("track_key")
df = df.groupby(["user_key", "track_key"]).size().reset_index(name="count")
users = df["user_key"].unique()
tracks = df["track_key"].unique()

user_to_idx = {u: i for i, u in enumerate(users)}
track_to_idx = {t: i for i, t in enumerate(tracks)}
idx_to_track = {i: u for u, i in track_to_idx.items()}
idx_to_user = {i: u for u, i in user_to_idx.items()}

rows = df["user_key"].map(user_to_idx).values
cols = df["track_key"].map(track_to_idx).values

data = df["count"].values

user_track_matrix = sp.csr_matrix(
    (data, (rows, cols)),
    shape=(len(users), len(tracks))
).tocsr()

In [9]:
model =AlternatingLeastSquares(
                factors=150, regularization=0.01, iterations=20
            )
model.fit(user_track_matrix)

/usr/local/lib/python3.12/dist-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/20 [00:00<?, ?it/s]

In [7]:
val_sessions = val_sessions.copy()
val_sessions = val_sessions[val_sessions["track_key"].apply(len) >= 2]

In [19]:
def evaluate_model(model, val_sessions, track_to_idx, idx_to_track, K=10):

    hits = 0
    total = 0

    for user,session in val_sessions[["user_key","track_key"]].values:
        if user not in user_to_idx:
          continue
        prefix = session[:-1]
        target = session[-1]

        cols = [track_to_idx[t] for t in prefix if t in track_to_idx]

        session_prefix_vec = sp.csr_matrix(
            (np.ones(len(cols)), ([0] * len(cols), cols)),
            shape=(1, len(track_to_idx))
        )
        user_id = user_to_idx[user]

        user_vec = user_track_matrix[user_id].copy()

        user_vec = user_vec + session_prefix_vec
        #user_vec = 0.7 * user_vec + 0.3 * session_prefix_vec

        recs, scores = model.recommend(user_id, user_vec,
                                               recalculate_user=True,
                                               N=10,
                                      filter_already_liked_items=True)


        target_idx = track_to_idx.get(target, None)
        if target_idx is not None and target_idx in recs:
            hits += 1

        total += 1

    return hits / total

In [12]:
item_factors = final_model.item_factors
print(np.linalg.norm(item_factors, axis=1)[:10])

[0.293469   0.31081942 0.37780407 0.42309204 0.1647353  0.10381573
 0.23125537 0.3727195  1.6813653  0.34907338]


In [20]:

alphas = [1, 2, 5, 10, 20, 40, 80]
ranks = [50, 100, 150, 200]
regs = [0.005, 0.01, 0.05, 0.1]

best_score = 0
best_params = {}
for i in range(20):

    rank = random.choice(ranks)
    reg = random.choice(regs)
    alpha = random.choice(alphas)

    model = AlternatingLeastSquares(
        factors=rank,
        regularization=reg,
        iterations=20
    )

    training = user_track_matrix.copy()


    training.data = 1 + alpha * training.data

    model.fit(training)

    score = evaluate_model(
        model,
        val_sessions,
        track_to_idx,
        idx_to_track,
        K=10
    )

    print(f"rank={rank}, reg={reg}, alpha={alpha} -> Hit@10={score:.4f}")

    if score > best_score:
        best_score = score
        best_params = {"factors": rank, "reg": reg, "alpha": alpha}

print(f"\nBest Params found: {best_params} with Hit@10: {best_score:.4f}")

  0%|          | 0/20 [00:00<?, ?it/s]

rank=100, reg=0.1, alpha=5 -> Hit@10=0.0010


  0%|          | 0/20 [00:00<?, ?it/s]

rank=50, reg=0.1, alpha=40 -> Hit@10=0.0021


  0%|          | 0/20 [00:00<?, ?it/s]

...

rank=100, reg=0.1, alpha=10 -> Hit@10=0.0021


  0%|          | 0/20 [00:00<?, ?it/s]

rank=50, reg=0.05, alpha=2 -> Hit@10=0.0010


  0%|          | 0/20 [00:00<?, ?it/s]

rank=150, reg=0.005, alpha=80 -> Hit@10=0.0010

Best Params found: {'factors': 50, 'reg': 0.01, 'alpha': 5} with Hit@10: 0.0031


In [30]:
final_model =AlternatingLeastSquares(
                factors=50, regularization=0.01, iterations=20
            )
training = user_track_matrix.copy()
training.data = 1 + 5 * training.data
final_model.fit(training)

  0%|          | 0/20 [00:00<?, ?it/s]

In [25]:
evaluate_model(
        final_model,
        test_sessions,
        track_to_idx,
        idx_to_track,
        K=10
    )

0.0010256410256410256

In [40]:
def give_recommendations(model, user,session_history, track_to_idx, idx_to_track, k=10):
  cols = [track_to_idx[t] for t in session_history if t in track_to_idx]

  session_prefix_vec = sp.csr_matrix(
        (np.ones(len(cols)), ([0] * len(cols), cols)),
        shape=(1, len(track_to_idx))
    )
  if user not in user_to_idx:
        recs, scores = model.recommend(0,session_prefix_vec,
                                               recalculate_user=True,
                                               N=k,
                                      filter_already_liked_items=True)
        target_idx = track_to_idx.get(target, None)
  else:
        user_id = user_to_idx[user]

        user_vec = user_track_matrix[user_id].copy()

        #user_vec = user_vec + session_prefix_vec
        user_vec = 0.7 * user_vec + 0.3 * session_prefix_vec

        recs, scores = model.recommend(user_id, user_vec,
                                               recalculate_user=True,
                                               N=k,
                                      filter_already_liked_items=True)


        rec_keys=[idx_to_track[r] for r in recs]
  return(rec_keys)
#example:
session=test_sessions.iloc[1]
user=session["user_key"]
prefix = session["track_key"][:-1]
target = session["track_key"][-1]
track_list=session["track_key"]
give_recommendations(final_model,user,prefix,track_to_idx,idx_to_track)


[np.int64(4485876),
 np.int64(1184528),
 np.int64(3754503),
 np.int64(13973443),
 np.int64(2315610),
 np.int64(3516308),
 np.int64(8396743),
 np.int64(7057302),
 np.int64(1224803),
 np.int64(8419145)]

,user_key,artist_key,track_key,session_start
global_session_id,,,,
user_000001_1067,user_000001,"[108268, 108268, 6380, 64403, 64403, 64403, 64...","[16738077, 16738077, 14941610, 1028583, 102858...",2009-05-02 14:19:06+00:00
user_000002_2016,user_000002,"[269155, 61793, 14376, 141662, 242430, 242430,...","[10972594, 1950039, 8490914, 242606, 9502222, ...",2009-04-26 14:37:07+00:00
user_000003_1535,user_000003,"[2865, 32372, 18674, 257937, 257937, 257937, 2...","[214250, 58711, 7240375, 9790735, 7458286, 852...",2009-04-10 16:20:25+00:00
user_000004_835,user_000004,"[7970, 7970, 7970, 189082, 189082]","[3123945, 16828400, 6150618, 8720592, 8746910]",2009-04-09 12:41:16+00:00
user_000005_1474,user_000005,"[158491, 217558, 269214, 30023, 202979, 202979...","[10950214, 9185745, 11927648, 2215927, 1049663...",2009-04-27 20:55:59+00:00
...,...,...,...,...
user_000996_106,user_000996,"[34989, 34989, 176444, 176444, 176444, 176444,...","[13685203, 13685203, 10525063, 3790858, 379085...",2007-12-03 20:27:08+00:00
user_000997_17,user_000997,"[112058, 112058, 112058, 112058, 112058, 11205...","[15721073, 4005262, 16905536, 1351905, 1067510...",2007-04-25 17:34:13+00:00
user_000998_3607,user_000998,"[264979, 139735, 139735, 139735, 139735, 13973...","[16872975, 4224688, 16083832, 16540001, 422468...",2009-04-29 19:11:59+00:00
